# Transfer Learning with TensorFlow Part 2: Fine-tuning

We have covered transfer learning feature extraction, now it's time to learn about a new kind of transfer learning: fine-tuning.

In [1]:
# Import helper functions we're going to use in this notebook.
from helper_functions import create_tensorboard_callback, plot_loss_curves, unzip_data, walk_through_dir

## Let's get some data

This time we're going to see how we can use the pretrained models within tf.keras.application

In [2]:
# Check out how many images and subdirectories are in our dataset
walk_through_dir("10_food_classes_10_percent")

There are 2 directories and 0 images in '10_food_classes_10_percent'.
There are 10 directories and 0 images in '10_food_classes_10_percent\test'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\chicken_curry'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\chicken_wings'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\fried_rice'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\grilled_salmon'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\hamburger'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\ice_cream'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\pizza'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\ramen'.
There are 0 directories and 250 images in '10_food_classes_10_percent\test\steak'.
There are 0 directories and 250 images in '10_food_classes_10_percent

In [3]:
# Create training and test directory paths
train_dir = "10_food_classes_10_percent/train"
test_dir = "10_food_classes_10_percent/test"

In [4]:
# data generators
import tensorflow as tf
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
train_data_10_percent = tf.keras.preprocessing.image_dataset_from_directory(directory=train_dir,
                                                                            image_size=IMG_SIZE,
                                                                            label_mode="categorical",
                                                                            batch_size=BATCH_SIZE)

test_data = tf.keras.preprocessing.image_dataset_from_directory(directory=test_dir,
                                                                image_size=IMG_SIZE,
                                                                label_mode="categorical",
                                                                batch_size=BATCH_SIZE)

Found 750 files belonging to 10 classes.
Found 2500 files belonging to 10 classes.


In [5]:
train_data_10_percent

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10), dtype=tf.float32, name=None))>

In [6]:
# Check out the class names of our dataset
train_data_10_percent.class_names

['chicken_curry',
 'chicken_wings',
 'fried_rice',
 'grilled_salmon',
 'hamburger',
 'ice_cream',
 'pizza',
 'ramen',
 'steak',
 'sushi']

In [7]:
# See an example of a batch of data
for images, labels in train_data_10_percent.take(1):
    print(images, labels)

tf.Tensor(
[[[[1.49857147e+02 7.59285736e+01 1.75714302e+01]
   [1.61760208e+02 7.99744949e+01 2.39030628e+01]
   [1.72913269e+02 8.11938782e+01 2.79795895e+01]
   ...
   [1.05086952e+02 6.13012161e+01 5.74441605e+01]
   [1.20055756e+02 7.09843140e+01 7.71986389e+01]
   [1.00315987e+02 4.83159866e+01 6.06731300e+01]]

  [[1.56948975e+02 8.69030609e+01 2.49030609e+01]
   [1.70219391e+02 9.05051041e+01 3.14336758e+01]
   [1.66729584e+02 7.75867386e+01 2.16581631e+01]
   ...
   [1.08423416e+02 6.36223717e+01 5.78112297e+01]
   [1.04806206e+02 5.48010864e+01 5.80205002e+01]
   [1.05719078e+02 5.27190819e+01 6.08619385e+01]]

  [[1.41336731e+02 7.55510254e+01 1.64081650e+01]
   [1.50147964e+02 7.37193909e+01 1.84336739e+01]
   [1.54591827e+02 6.95459137e+01 1.66173477e+01]
   ...
   [1.29775848e+02 8.07299194e+01 7.27044678e+01]
   [1.03398109e+02 5.17705231e+01 4.82297249e+01]
   [1.19974701e+02 6.55461273e+01 6.53318405e+01]]

  ...

  [[1.81377411e+01 1.13774109e+00 1.87092133e+01]
   [1

## Model 0: Building a transfer learning feature extraction model using the Keras Functional API

The sequential API is straight-forward, it runs our layers is sequential order.
But the functional API gives us more flexibility with our models.

In [8]:
# 1. Create base model with tf.keras.applications
base_model = tf.keras.applications.EfficientNetB0(include_top=False)

# 2. Freeze the base model (so the underlying pre-trained patterns aren't updated during training)
base_model.trainable = False

# 3. Create inputs into our model
inputs = tf.keras.layers.Input(shape=(224, 224, 3), name="input_layer")

# 4. If using a model like ResNet50V2 you will need to normalize inputs
# x = tf.keras.layers.experimental.preprocessing.Rescaling(1./255)(inputs)

# 5. Pass the inputs to the base model
x = base_model(inputs)
print(f"shape after passing inputs through base model: {x.shape}")

# 6. Average pol the outputs of the base model (aggregate all the most important information, reduce number of conputations)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
print(f"Shape after GlobalAveragePooling2D: {x.shape}")

# 7. Create the output activaiton layer
outputs = tf.keras.layers.Dense(10, activation="softmax", name="output_layer")(x)

# 8. Combine the inputs with the outputs into a model
model_0 = tf.keras.Model(inputs, outputs)

# 9. Compile the model
model_0.compile(loss="categorical_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

# 10. Fit the model and save its history
history_10_percent = model_0.fit(train_data_10_percent,
                                 epochs=5,
                                 steps_per_epoch=len(train_data_10_percent),
                                 validation_data=test_data,
                                 validation_steps=int(0.25 * len(test_data)),
                                 callbacks=[create_tensorboard_callback(dir_name="transfer_learning",
                                                                        experiment_name="10_percent_feature_extraction")])



16705208/16705208 [==============================] - 6s 0us/step
shape after passing inputs through base model: (None, 7, 7, 1280)
Shape after GlobalAveragePooling2D: (None, 1280)
Saving TensorBoard log files to: transfer_learning/10_percent_feature_extraction/20240331-145131
Epoch 1/5


24/24 [==============================] - 55s 2s/step - loss: 1.8410 - accuracy: 0.4267 - val_loss: 1.2901 - val_accuracy: 0.7385
Epoch 2/5
24/24 [==============================] - 43s 2s/step - loss: 1.0806 - accuracy: 0.7427 - val_loss: 0.8592 - val_accuracy: 0.8207
Epoch 3/5
24/24 [==============================] - 36s 2s/step - loss: 0.7797 - accuracy: 0.8440 - val_loss: 0.7106 - val_accuracy: 0.8355
Epoch 4/5
24/24 [==============================] - 31s 1s/step - loss: 0.6381 - accuracy: 0.8520 - val_loss: 0.6213 - val_accuracy: 0.8487
Epoch 5/5
24/24 [==============================] - 31s 1s/step - loss: 0.5515 - accuracy: 0.8733 - val_loss: 0.5872 - val_accuracy: 0.8438
